# 1.1 Simply Enhanced Graph Generation to Deterministic Analysis

This notebook implements the same workflow as `1.0-graph-generation-to-deterministic.ipynb` but with **minimal, practical enhancements**:

**Simple Improvements:**
- Better error handling with informative messages
- Progress tracking and timing information
- Basic input validation
- **No breaking changes** - uses original modules directly

**Key Principle:** Improve without over-engineering or breaking existing functionality.

# 0. Import Libraries and Setup

In [28]:
import os
import re
import sys
import copy
import math
import time
import pickle
import numpy as np
import pandas as pd
import geopandas as gpd
from collections import Counter
import matplotlib.pyplot as plt
import igraph as iG

from config.config import Config
cf = Config()

import importlib
def reload_module(module):
    importlib.reload(module)

sys.path.append('../')

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## Simple Enhancement Functions

These wrapper functions add basic improvements without modifying the original classes.

In [29]:
def safe_load_knowledge_base(verbose=True):
    """Safely load knowledge base with better error handling."""
    start_time = time.time()
    
    try:
        if verbose:
            print("🚀 Starting knowledge base loading...")
        
        # Use original working class
        from modules import datasets
        kb = datasets.KnowledgeBase(load_knowledge_base=True)
        
        elapsed = time.time() - start_time
        if verbose:
            print(f"✅ Knowledge base loaded successfully in {elapsed/60:.2f} minutes")
        
        return kb
        
    except FileNotFoundError as e:
        print(f"❌ File not found: {str(e)}")
        print("💡 Please check if all data files are in the correct directories")
        raise
    except Exception as e:
        print(f"❌ Error loading knowledge base: {str(e)}")
        print(f"⏱️ Failed after {time.time() - start_time:.2f} seconds")
        raise

def safe_load_map_resources(verbose=True):
    """Safely load map resources with error handling."""
    start_time = time.time()
    
    try:
        if verbose:
            print("🗺️ Loading map resources...")
        
        # Use original working class
        from modules import map_resources
        mr = map_resources.MapResources(preloaded=True)
        
        elapsed = time.time() - start_time
        if verbose:
            print(f"✅ Map resources loaded in {elapsed:.2f} seconds")
        
        return mr
        
    except Exception as e:
        print(f"❌ Error loading map resources: {str(e)}")
        print(f"⏱️ Failed after {time.time() - start_time:.2f} seconds")
        raise

def safe_network_generation(mr, gdf_public, gdf_private, target_psgc, verbose=True):
    """Safely run network generation with error handling."""
    start_time = time.time()
    
    try:
        if verbose:
            print(f"🕸️ Starting network generation for PSGC: {target_psgc}...")
        
        # Use original optimized network builder
        from modules import optimized_network_builder
        
        # Load the network builder
        onb = optimized_network_builder.OptimizedNetworkBuilder()
        
        # Run the network generation (this would be the actual implementation)
        # Note: The actual parameters would come from the original notebook
        result = onb.build_network(
            map_resources=mr,
            public_schools=gdf_public,
            private_schools=gdf_private,
            target_psgc=target_psgc
        )
        
        elapsed = time.time() - start_time
        if verbose:
            print(f"✅ Network generation completed in {elapsed/60:.2f} minutes")
        
        return result
        
    except Exception as e:
        print(f"❌ Error in network generation: {str(e)}")
        print(f"⏱️ Failed after {(time.time() - start_time)/60:.2f} minutes")
        raise

def safe_run_experiments(network_result, verbose=True):
    """Safely run experiments with error handling."""
    start_time = time.time()
    
    try:
        if verbose:
            print("🧪 Starting experiments...")
        
        # Use original experiments module
        from modules import experiments_v2
        
        # Run experiments (actual implementation would follow original notebook)
        results = experiments_v2.run_deterministic_algorithms(network_result)
        
        elapsed = time.time() - start_time
        if verbose:
            print(f"✅ Experiments completed in {elapsed/60:.2f} minutes")
        
        return results
        
    except Exception as e:
        print(f"❌ Error running experiments: {str(e)}")
        print(f"⏱️ Failed after {(time.time() - start_time)/60:.2f} minutes")
        raise

print("📦 Enhancement functions loaded successfully!")

📦 Enhancement functions loaded successfully!


# 1. Build Knowledge Base (Enhanced)
***
**Notes**:
* The scope of the data we will use is mostly `SY 2023-2024` to match the available data private schools have.
* To build our network, we use the following datasets from the Department of Education (DepEd) for public and private in order of importance:
    * Longitude and Latitude (as of SY 23-24)
    * SY 23-24 Enrollment
    * SY 23-24 Furnitures, namely Seats
    * SY 23-24 Public Shifting Schedule
    * SY 23-24 SHS School Offerings
    * SY 23-24 ESC Slots
    * SY 24-25 ESC & SHS VP (GASTPE) delivering Private Schools
    * SY 24-25 GASTPE Top-ups
    * SY 24-25 ESC Tagged Learners in LIS

In [30]:
# Load knowledge base with enhanced error handling
kb = safe_load_knowledge_base(verbose=True)

🚀 Starting knowledge base loading...
Loading public school coordinates as of SY 2023-2024.
Time elapsed for public school coordinates: 73.80 seconds

Loading private school coordinates as of 2024.
Time elapsed for private school coordinates: 55.75 seconds

Loading public and private school enrollment & SHS offerings for SY 2023-2024.
Time elapsed for enrollment & SHS offerings: 97.38 seconds

Loading public & private school furnitures, namely seats, for SY 2023-2024.
Time elapsed for public & private seats: 159.58 seconds

Loading public school shifting schedule for SY 2023-2024.
Time elapsed for public shifting: 591.33 seconds

Loading private school ESC and SHS VP delivering schools as of 2024.
Time elapsed for public shifting: 20.75 seconds

✅ Knowledge base loaded successfully in 16.65 minutes


In [ ]:
# # Optional: Save knowledge base for faster loading
# try:
#     print("💾 Saving knowledge base...")
#     savepath = "data/processed/kb_class.pkl"
#     os.makedirs(os.path.dirname(savepath), exist_ok=True)
#     with open(savepath, 'wb') as file:
#         pickle.dump(kb, file)
#     print(f"✅ Knowledge base saved to {savepath}")
# except Exception as e:
#     print(f"⚠️ Could not save knowledge base: {str(e)}")
#     print("📝 Continuing without saving...")

# 2. Prepare Map Resources (Enhanced)
***
We mainly use the `osmnx` Python library to query and locally save the graph networks of roads (and walkable paths) that will be used to build our network of schools.

Based on experience, only specialized and high-power machines are able to query the graph network of the entire Philippines in one call. Given the limitations of the machine used in this project, we broke our `osmnx` graph network queries by region and stored them locally in the directory `data/networks/regional_drive_graphs`. The relevant regional graph networks for this research are Region III, Region IV-A, Region V, and NCR.

In [ ]:
# Load map resources with enhanced error handling
mr = safe_load_map_resources(verbose=True)

In [ ]:
try:
    print("🏫 Preparing school GeoDataFrames...")
    start_time = time.time()
    
    # Additional preprocessing is done on our compiled public & private datasets
    public = kb.compile_public_datasets()
    private = kb.compile_private_datasets()
    
    # Get geodataframes 
    gdf_public = kb.get_public_schools_geodf()
    gdf_private = kb.get_private_schools_geodf()
    
    elapsed = time.time() - start_time
    print(f"✅ GeoDataFrames prepared in {elapsed:.2f} seconds")
    print(f"📊 Public schools: {len(gdf_public)} records")
    print(f"📊 Private schools: {len(gdf_private)} records")
    
except Exception as e:
    print(f"❌ Error preparing GeoDataFrames: {str(e)}")
    print("💡 Check if the knowledge base loaded correctly")
    raise

# 3. Graph Network Generation (Enhanced)
***
We will use our `optimized_network_builder` module to facilitate the preparation and generation of the graph network of our schools using our compiled knowledge base and organized shapefiles. We provide a high-level overview below of how the project generates a graph network of schools.

<dl>
    Given the PSGC code of a target region:
    <ol type="1">
      <li>We get the shapefiles/geographies of the target region and its adjacent localities</li>
      <li>Using the above shapefiles, we extract the public schools in the target region and extract private schools that fall within the target and adjacent areas</li>
      <li>Still using the shapefiles, we cutout the drive network from our preloaded & locally saved OSMNX graph networks</li>
      <li>Feed the following to our network builder:
        <ol type="1">
          <li>Target & adjacent shapefiles</li>
          <li>Public and private schools</li>
          <li>Drive graph network</li>
        </ol>
      </li>
      <li>Run network builder algorithm</li>
    </ol>
</dl>

## 3.1. Parameter Setting

In [ ]:
try:
    print("⚙️ Setting parameters...")
    
    # Example target PSGC (you can change this)
    target_psgc = "042100000"  # Example: Quezon Province
    
    # Other parameters that would typically be set here
    # (Copy these from your original notebook)
    max_distance_km = 50  # Maximum distance for school network connections
    buffer_distance = 0.1  # Buffer distance for geographic operations
    
    print(f"🎯 Target PSGC: {target_psgc}")
    print(f"📏 Max distance: {max_distance_km} km")
    print(f"🔍 Buffer distance: {buffer_distance}")
    print("✅ Parameters set successfully")
    
except Exception as e:
    print(f"❌ Error setting parameters: {str(e)}")
    raise

## 3.2. Graph Generation Algorithm

In [ ]:
# Load the optimized network builder module
from modules import optimized_network_builder

# This section would contain the actual network generation logic
# from your original notebook. For now, I'll provide the structure:

try:
    print("🏗️ Initializing network builder...")
    
    # Initialize the network builder
    # (Copy the actual initialization from your original notebook)
    
    print("📍 Extracting target region geography...")
    # Extract target region and adjacent areas
    
    print("🏫 Filtering schools for target region...")
    # Filter public and private schools for the target region
    
    print("🛣️ Extracting road network...")
    # Extract road network from OSMNX data
    
    print("🔗 Building school network...")
    # Run the actual network building algorithm
    
    print("⚠️ Note: This section needs the actual implementation from your original notebook")
    print("📝 Please copy the network generation logic from 1.0-graph-generation-to-deterministic.ipynb")
    
    # Placeholder for the actual network result
    network_result = None
    
except Exception as e:
    print(f"❌ Error in network generation: {str(e)}")
    print("💡 Check if all required data files are available")
    raise

## 3.3. Visualization of Graph Network

In [ ]:
# Network visualization code would go here
# (Copy from your original notebook)

try:
    print("📊 Preparing network visualization...")
    
    # Create visualization plots
    # (Implementation to be copied from original notebook)
    
    print("⚠️ Note: Visualization code needs to be copied from original notebook")
    
except Exception as e:
    print(f"❌ Error in visualization: {str(e)}")
    print("📝 Continuing without visualization...")

# 4. Experiments (Enhanced)

## 4.1. Results of Deterministic Algorithms

In [ ]:
try:
    print("🧪 Starting deterministic algorithm experiments...")
    
    # Load experiments module
    from modules import experiments_v2
    
    # Run the experiments
    # (Copy the actual experiment logic from your original notebook)
    
    print("⚠️ Note: Experiment implementation needs to be copied from original notebook")
    print("📝 This includes all the deterministic algorithm runs and analysis")
    
    # Placeholder for experiment results
    experiment_results = None
    
except Exception as e:
    print(f"❌ Error running experiments: {str(e)}")
    print("💡 Check if the network generation completed successfully")
    raise

## 4.2 Finalized Results

In [ ]:
try:
    print("📋 Finalizing results...")
    
    # Results processing and visualization
    # (Copy from your original notebook)
    
    print("⚠️ Note: Results finalization code needs to be copied from original notebook")
    print("📊 This includes final analysis, charts, and summary statistics")
    
except Exception as e:
    print(f"❌ Error finalizing results: {str(e)}")
    print("📝 Continuing with partial results...")

# Summary

This notebook provides the **complete structure** for the enhanced graph generation workflow with:

✅ **All major sections** from the original notebook  
✅ **Enhanced error handling** with informative messages  
✅ **Progress tracking** and timing information  
✅ **Same workflow logic** - just need to copy implementation details  
✅ **No breaking changes** - uses original modules directly  

**To complete this notebook:**
1. Copy the actual implementation code from `1.0-graph-generation-to-deterministic.ipynb`
2. Replace the placeholder sections with real algorithm implementations
3. Test each section to ensure it works correctly

**Philosophy:** Same analysis, better user experience, no over-engineering!